## Metadata Query Agent — AgentCore On-Demand Evaluation

This notebook runs **on-demand evaluation** of the Metadata Query Agent deployed on **Amazon Bedrock AgentCore Runtime**. Unlike [notebook 2](./2_metadata_query_agent_evaluation.ipynb) which evaluates the agent locally using Strands in-memory telemetry, this notebook:

1. **Restores agent configuration** from notebook 2 (`metadata_id` via `%store`)
2. **Looks up the AgentCore Runtime ARN** from CloudFormation stack outputs
3. **Invokes the agent on AgentCore Runtime** for each test case — generating OTEL traces in AgentCore Observability
4. **Runs on-demand evaluation** using the `bedrock_agentcore_starter_toolkit` with built-in evaluators

### On-Demand vs Online Evaluation

| Aspect | On-Demand (this notebook) | Online (notebook 4) |
|:-------|:--------------------------|:--------------------|
| Trigger | Developer-initiated | Continuous / automatic |
| Scope | Selected sessions / traces | All production traffic (sampled) |
| Use case | Targeted investigation, regression testing | Production monitoring |

### Prerequisites

- **Notebook 1** (`1_metadata_agent_evaluation.ipynb`) must have been run to `%store metadata_id`
- **AgentCore stack** must be deployed (`cdk deploy semantic-layer-agentcore`)
- **Metadata Query Agent** must be running on AgentCore Runtime

In [1]:
# Prereq:
# !pip install bedrock-agentcore-starter-toolkit==0.3.3

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac-demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

✓ AWS Credentials Verified:
  Profile: huthmac-demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
# ── Restore metadata_id from notebook 1 (via %store) ──
%store -r metadata_id
EVAL_ID = metadata_id
print(f"✓ Loaded EVAL_ID from %store: {EVAL_ID}")

# ── Look up Metadata Query Runtime ARN from CloudFormation stack outputs ──
cfn = session.client('cloudformation', region_name=region)
stack_name = f'{PROJECT_NAME}-agentcore'
outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

METADATA_QUERY_RUNTIME_ARN = output_map['MetadataQueryRuntimeArn']
# Extract agent_id from ARN: arn:aws:bedrock-agentcore:region:account:agentruntime/<id>
agent_id = METADATA_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"\n✓ AgentCore Runtime details (from stack '{stack_name}'):")
print(f"  Runtime ARN: {METADATA_QUERY_RUNTIME_ARN}")
print(f"  Agent ID:    {agent_id}")

# ── Load evaluation dataset ──
with open('../data/eval/metadata_query_agent_evaluation_dataset.json', 'r') as f:
    test_cases = json.load(f)

print(f"\n✓ Loaded {len(test_cases)} test case(s):")
for tc in test_cases:
    print(f"  [{tc['id']}] ({tc['category']}): {tc['query']}")

✓ Loaded EVAL_ID from %store: semantic-rag-single_table_basic-8f7bcf77

✓ AgentCore Runtime details (from stack 'semantic-layer-agentcore'):
  Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:381492284087:runtime/semantic_layer_metadata_query-itnnG8AGV6
  Agent ID:    semantic_layer_metadata_query-itnnG8AGV6

✓ Loaded 1 test case(s):
  [test1] (admin_codes_test1): How many unique admincodes exist?


### Invoke Agent on AgentCore Runtime

Before running on-demand evaluation, we need to invoke the agent to generate traces in AgentCore Observability. Each invocation creates a session with OTEL spans that the evaluation service can analyze.

We use a single `runtimeSessionId` for all test cases so the entire evaluation run is grouped in one session.

In [4]:
agentcore_client = session.client('bedrock-agentcore', region_name=region)

# Use a single session ID for this evaluation run
eval_session_id = str(uuid.uuid4())
print(f"Evaluation session ID: {eval_session_id}")
print(f"{'='*70}\n")

invocation_results = []

for tc in test_cases:
    payload = json.dumps({'question': tc['query'], 'id': EVAL_ID}).encode('utf-8')

    print(f"[{tc['id']}] Invoking: {tc['query']}")
    start = time.time()

    response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=METADATA_QUERY_RUNTIME_ARN,
        runtimeSessionId=eval_session_id,
        payload=payload,
        qualifier='DEFAULT',
    )

    # Read streaming response
    content = [chunk.decode('utf-8') for chunk in response.get('response', [])]
    response_text = ''.join(content)
    duration = time.time() - start

    try:
        response_data = json.loads(response_text)
    except json.JSONDecodeError:
        response_data = {'result': response_text}

    invocation_results.append({
        'test_id': tc['id'],
        'query': tc['query'],
        'response': response_data,
        'duration': duration,
    })

    answer_preview = str(response_data)[:120]
    print(f"  ✓ {duration:.1f}s — {answer_preview}...")
    print()

print(f"{'='*70}")
print(f"✓ All {len(test_cases)} test case(s) invoked on AgentCore Runtime")
print(f"  Session ID: {eval_session_id}")
print(f"  Traces are now available in AgentCore Observability")

Evaluation session ID: e13b2615-89bc-4273-aa02-bbcfff43c047

[test1] Invoking: How many unique admincodes exist?
  ✓ 17.6s — {'answer': 'There are **10 unique admin codes** in the `admin_codes` table, as identified by the distinct values of the ...

✓ All 1 test case(s) invoked on AgentCore Runtime
  Session ID: e13b2615-89bc-4273-aa02-bbcfff43c047
  Traces are now available in AgentCore Observability


In [5]:
# ── Wait for ALL OTEL spans to be indexed in aws/spans ────────────────────────
# CWL Insights is eventually consistent: spans arrive in batches and queries
# during active ingestion return partial results, making poll-based readiness
# detection unreliable.
#
# Empirical observation: for this agent (~12 Strands spans per invocation)
# all spans are fully indexed within 90s of invocation completion.
#
# Strategy: sleep 90s (unconditional), then do a single confirmation that
# the invoke_agent Strands Agents span is present before proceeding.
import boto3 as _boto3, time as _time

_logs = _boto3.client('logs', region_name=region)
_agent_id = agent_id

_fixed_wait = 90   # seconds — empirically safe for full span indexing
print(f"Sleeping {_fixed_wait}s for CWL Insights to finish indexing all spans ...")

for _i in range(_fixed_wait // 10):
    _time.sleep(10)
    print(f"  [{(_i+1)*10}s] ...")

# Confirmation: verify invoke_agent span is present before evaluation
print("Confirming 'invoke_agent Strands Agents' span is indexed ...")
_confirmed = False
for _attempt in range(4):
    try:
        _resp = _logs.start_query(
            logGroupName='aws/spans',
            startTime=int(_time.time()) - 600,
            endTime=int(_time.time()),
            queryString=(
                f"fields spanId | filter attributes.session.id = '{eval_session_id}' "
                f"| filter name = 'invoke_agent Strands Agents' "
                f"| parse resource.attributes.cloud.resource_id \"runtime/*/\" as parsedAgentId "
                f"| filter parsedAgentId = '{_agent_id}' | limit 1"
            ),
        )
        for _ in range(20):
            _time.sleep(1)
            _r = _logs.get_query_results(queryId=_resp['queryId'])
            if _r['status'] == 'Complete':
                if _r.get('results'):
                    _confirmed = True
                break
    except Exception as _e:
        print(f"  confirm error: {_e}")
    if _confirmed:
        break
    print(f"  invoke_agent not yet visible, waiting 15s more ...")
    _time.sleep(15)

if _confirmed:
    print(f"✓ All spans ready — proceeding to evaluation")
else:
    print(f"⚠️  invoke_agent span not confirmed — evaluation may fail.")


Sleeping 90s for CWL Insights to finish indexing all spans ...
  [10s] ...
  [20s] ...
  [30s] ...
  [40s] ...
  [50s] ...
  [60s] ...
  [70s] ...
  [80s] ...
  [90s] ...
Confirming 'invoke_agent Strands Agents' span is indexed ...
✓ All spans ready — proceeding to evaluation


### Initialize AgentCore Evaluation Client

The `bedrock_agentcore_starter_toolkit` simplifies on-demand evaluation by automatically extracting traces from a session and running evaluators against them.

In [6]:
from bedrock_agentcore_starter_toolkit import Evaluation

eval_client = Evaluation(region=region)
print(f"✓ AgentCore Evaluation client initialized (region={region})")

/Users/huthmac/Documents/AWS/00_workspace/semantic-layer/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


✓ AgentCore Evaluation client initialized (region=us-east-1)


### Run On-Demand Evaluations

We evaluate the agent's session traces using four built-in evaluators at different levels:

| Evaluator | Level | What it measures |
|:----------|:------|:-----------------|
| `Builtin.GoalSuccessRate` | Session | Did the agent achieve the user's goal end-to-end? |
| `Builtin.Correctness` | Trace | Is the agent's answer factually correct? |
| `Builtin.ToolSelectionAccuracy` | Span | Did the agent pick the right tools in the right order? |
| `Builtin.ToolParameterAccuracy` | Span | Were tool parameters (SQL, database, catalog) correct? |

### Goal Success Rate

In [7]:
print(f"Evaluating session {eval_session_id} for GoalSuccessRate...")
goal_success_results = eval_client.run(
    agent_id=agent_id,
    session_id=eval_session_id,
    evaluators=["Builtin.GoalSuccessRate"],
)
print("✓ GoalSuccessRate evaluation complete")

Evaluating session e13b2615-89bc-4273-aa02-bbcfff43c047 for GoalSuccessRate...


Evaluating session: e13b2615-89bc-4273-aa02-bbcfff43c047

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.GoalSuccessRate

/Users/huthmac/Documents/AWS/00_workspace/semantic-layer/venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

✓ GoalSuccessRate evaluation complete


The results include the evaluation label, numeric value, explanation, token usage, and span context.

In [8]:
for result in goal_success_results.results:
    info = f"""
**Goal Success: {result.label}** ({result.value})

**Explanation:** {result.explanation}

Token Usage: {result.token_usage}
"""
    display(Markdown(info))


**Goal Success: None** (None)

**Explanation:** 

Token Usage: None


### Correctness

Correctness is evaluated at the **trace level** — each invocation trace gets its own score.

In [9]:
print(f"Evaluating session {eval_session_id} for Correctness...")
correctness_results = eval_client.run(
    agent_id=agent_id,
    session_id=eval_session_id,
    evaluators=["Builtin.Correctness"],
)
print("✓ Correctness evaluation complete")

Evaluating session e13b2615-89bc-4273-aa02-bbcfff43c047 for Correctness...


Evaluating session: e13b2615-89bc-4273-aa02-bbcfff43c047

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.Correctness

✓ Correctness evaluation complete


Each trace gets its own correctness evaluation result.

In [10]:
for result in correctness_results.results:
    info = f"""
**Correctness: {result.label}** ({result.value})

**Explanation:** {result.explanation}

Token Usage: {result.token_usage}
"""
    display(Markdown(info))
    print("=" * 60)


**Correctness: None** (None)

**Explanation:** 

Token Usage: None


### Tool Selection & Parameter Accuracy

Both metrics are evaluated at the **span level** — each tool call span is scored individually. We run them together in a single evaluation request.

In [11]:
print(f"Evaluating session {eval_session_id} for ToolParameterAccuracy + ToolSelectionAccuracy...")
tool_results = eval_client.run(
    agent_id=agent_id,
    session_id=eval_session_id,
    evaluators=["Builtin.ToolParameterAccuracy", "Builtin.ToolSelectionAccuracy"],
)
print("✓ Tool evaluation complete")

Evaluating session e13b2615-89bc-4273-aa02-bbcfff43c047 for ToolParameterAccuracy + ToolSelectionAccuracy...


Evaluating session: e13b2615-89bc-4273-aa02-bbcfff43c047

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.ToolParameterAccuracy, Builtin.ToolSelectionAccuracy

✓ Tool evaluation complete


Since we evaluated with two metrics in one run, we use `evaluator_name` to distinguish results.

In [12]:
for result in tool_results.results:
    info = f"""
**Metric:** {result.evaluator_name}
**Value:** {result.label} ({result.value})

**Explanation:** {result.explanation}

Token Usage: {result.token_usage}
"""
    display(Markdown(info))
    print("=" * 60)


**Metric:** Builtin.ToolParameterAccuracy
**Value:** None (None)

**Explanation:** 

Token Usage: None



**Metric:** Builtin.ToolParameterAccuracy
**Value:** None (None)

**Explanation:** 

Token Usage: None



**Metric:** Builtin.ToolParameterAccuracy
**Value:** None (None)

**Explanation:** 

Token Usage: None



**Metric:** Builtin.ToolSelectionAccuracy
**Value:** None (None)

**Explanation:** 

Token Usage: None



**Metric:** Builtin.ToolSelectionAccuracy
**Value:** None (None)

**Explanation:** 

Token Usage: None



**Metric:** Builtin.ToolSelectionAccuracy
**Value:** None (None)

**Explanation:** 

Token Usage: None


### Combined Evaluation — All Built-in Evaluators

Run all four built-in evaluators in a single request and save results to a file.

In [13]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'../data/eval/results/ac_ondemand_eval_{timestamp}.json'

print(f"Running all built-in evaluators and saving to {output_file}...")
all_results = eval_client.run(
    agent_id=agent_id,
    session_id=eval_session_id,
    evaluators=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolParameterAccuracy",
        "Builtin.ToolSelectionAccuracy",
    ],
    output=output_file,
)
print(f"✓ Combined evaluation complete — {len(all_results.results)} result(s) saved")

Running all built-in evaluators and saving to ../data/eval/results/ac_ondemand_eval_20260323_114553.json...


Evaluating session: e13b2615-89bc-4273-aa02-bbcfff43c047

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.GoalSuccessRate, Builtin.Correctness, Builtin.ToolParameterAccuracy, 
Builtin.ToolSelectionAccuracy

✓ Results saved to: ../data/eval/results/ac_ondemand_eval_20260323_114553.json

✓ Input data saved to: ../data/eval/results/ac_ondemand_eval_20260323_114553_input.json

✓ Combined evaluation complete — 8 result(s) saved


### Results Summary

In [14]:
# Build summary table from all_results
summary_data = []
for result in all_results.results:
    summary_data.append({
        'Evaluator': result.evaluator_name,
        'Label': result.label,
        'Score': result.value,
        'Explanation': result.explanation[:120] + '...' if len(str(result.explanation)) > 120 else result.explanation,
    })

df_summary = pd.DataFrame(summary_data)
print(f"Session: {eval_session_id}")
print(f"Agent:   {agent_id}")
print(f"EVAL_ID: {EVAL_ID}")
print(f"\n{'='*70}")
display(df_summary)

# Average score
scores = [r.value for r in all_results.results if r.value is not None]
if scores:
    print(f"\nAverage score: {sum(scores) / len(scores):.3f}")

Session: e13b2615-89bc-4273-aa02-bbcfff43c047
Agent:   semantic_layer_metadata_query-itnnG8AGV6
EVAL_ID: semantic-rag-single_table_basic-8f7bcf77



,Evaluator,Label,Score,Explanation
0,Builtin.GoalSuccessRate,Yes,1.0,The user's goal was to determine how many uniq...
1,Builtin.Correctness,Perfectly Correct,1.0,The task asks 'How many unique admincodes exis...
2,Builtin.ToolParameterAccuracy,Yes,1.0,The tool-call is to 'retrieve_kb_context' with...
3,Builtin.ToolParameterAccuracy,Yes,1.0,The tool-call to 'disambiguate_query_terms' ha...
4,Builtin.ToolParameterAccuracy,Yes,1.0,Analyzing each parameter in the execute_sql_qu...
5,Builtin.ToolSelectionAccuracy,Yes,1.0,The user asks 'How many unique admincodes exis...
6,Builtin.ToolSelectionAccuracy,Yes,1.0,The user asked 'How many unique admincodes exi...
7,Builtin.ToolSelectionAccuracy,Yes,1.0,The user asked 'How many unique admincodes exi...



Average score: 1.000


### Store Session ID for Downstream Notebooks

Save the evaluation session ID so notebook 4 (online evaluation) can reference it.

In [15]:
ondemand_session_id = eval_session_id
ondemand_agent_id = agent_id
%store ondemand_session_id
%store ondemand_agent_id

print(f"✓ Stored for downstream notebooks:")
print(f"  ondemand_session_id = {ondemand_session_id}")
print(f"  ondemand_agent_id   = {ondemand_agent_id}")
print(f"\n  Results file: {output_file}")

Stored 'ondemand_session_id' (str)
Stored 'ondemand_agent_id' (str)
✓ Stored for downstream notebooks:
  ondemand_session_id = e13b2615-89bc-4273-aa02-bbcfff43c047
  ondemand_agent_id   = semantic_layer_metadata_query-itnnG8AGV6

  Results file: ../data/eval/results/ac_ondemand_eval_20260323_114553.json


## Summary

This notebook demonstrated **on-demand evaluation** of the Metadata Query Agent on AgentCore Runtime:

1. **Restored** `metadata_id` from notebook 1 and the AgentCore Runtime ARN from CloudFormation
2. **Invoked** the agent for each test case — generating OTEL traces in AgentCore Observability
3. **Evaluated** the session using four built-in metrics:
   - `GoalSuccessRate` (session-level)
   - `Correctness` (trace-level)
   - `ToolParameterAccuracy` (span-level)
   - `ToolSelectionAccuracy` (span-level)

### Comparison with Notebook 2

| Aspect | Notebook 2 (Strands local) | Notebook 3 (AgentCore on-demand) |
|:-------|:---------------------------|:---------------------------------|
| Agent execution | Local Python process | AgentCore Runtime (deployed) |
| Telemetry | In-memory OTEL exporter | CloudWatch + AgentCore Observability |
| Evaluators | Strands `HelpfulnessEvaluator`, etc. | `Builtin.GoalSuccessRate`, etc. |
| Use case | Development iteration | Pre-production / staging validation |

### Next Steps
- Run **notebook 4** for online (continuous) evaluation of production traffic
- Expand the evaluation dataset with more test cases
- Add custom evaluators for domain-specific quality criteria